# NSE 236 Final Report

### Part 1: Two-Group Diffusion Solver

This section of the report aims to facilitate understanding of our two-group neutron diffusion code, designed to iteratively solve the following equations in one dimension:

**Fast Group:**
$$-\frac{\partial}{\partial z}\left(D_1(z) \frac{\partial \phi_1}{\partial z}\right) + \Sigma_{r,1}(z)\phi_1 = \frac{1}{k} S_f(z)$$
**Thermal Group:**
$$-\frac{\partial}{\partial z}\left(D_2(z) \frac{\partial \phi_2}{\partial z}\right) + \Sigma_{a,2}(z)\phi_2 = \Sigma_{s,1\rightarrow2}(z)\phi_1$$

The one-group code was modified to welcome a second set of the same variables corresponding to the thermal group, as well as the thermal source term ($\Sigma_{s,1\rightarrow2}$, i.e. thermal downscattering).

### Libraries:

All of the libraries from the one-group code were conserved. The math library was added.

In [76]:
import  numpy                   as      np
import  matplotlib.pyplot       as      plt
from    scipy.sparse            import  diags
from    scipy.sparse.linalg     import  spsolve
from    math                    import  *

### Functions:

**expand_regions:**

This function takes the variables & coefficients associated with the reactor & extends them to a collection of one-dimensional cells for iteration & graphing.

In [77]:
def expand_regions(regions):
    D1, D2, Sa1, Sa2, Ss12, nSf1, nSf2, dx, labels = [], [], [], [], [], [], [], [], []

    for reg in regions:
        Nc = reg['Nc']
        D1.extend([reg['D1']] * Nc)
        D2.extend([reg['D2']] * Nc)
        Sa1.extend([reg['Sa1']] * Nc)
        Sa2.extend([reg['Sa2']] * Nc)
        Ss12.extend([reg['Ss12']] * Nc)
        nSf1.extend([reg['nSf1']] * Nc)
        nSf2.extend([reg['nSf2']] * Nc)
        dx.extend([reg['dx']] * Nc)
        labels.extend([reg['label']] * Nc)
    
    D1          = np.array(D1,      dtype=float)
    D2          = np.array(D2,      dtype=float)
    Sa1         = np.array(Sa1,     dtype=float)
    Sa2         = np.array(Sa2,     dtype=float)
    Ss12        = np.array(Ss12,    dtype=float)
    nSf1        = np.array(nSf1,    dtype=float)
    nSf2        = np.array(nSf2,    dtype=float)
    dx          = np.array(dx,      dtype=float)

    x_edges     = np.concatenate(([0.0], np.cumsum(dx)))
    x_centers   = 0.5 * (x_edges[:-1] + x_edges[1:])

    return x_centers, D1, D2, Sa1, Sa2, Ss12, nSf1, nSf2, dx, labels

**build_loss_matrix:**

This function produces the tridiagonal matrix , with the assumption of zero incident partial current at the top and bottom of the reactor.

In [78]:
def build_loss_matrix(D, Sig_r, dx):
    """
    Builds tridiagonal matrix for a 1D diffusion system.
    Applies vacuum (zero incident partial current) boundary conditions
    at both the top and the bottom of the reactor.
    """
    N = len(D)

    lower = np.zeros(N)
    diag = np.zeros(N)
    upper = np.zeros(N)

    # Interior interface conductances
    F = np.zeros(N - 1) # F array is used to ensure the leakage is conserved across the interfaces
    for i in range(N - 1):
        F[i] = 1.0 / (0.5 * dx[i] / D[i] + 0.5 * dx[i+1] / D[i+1])

    for i in range(N):
        diag[i] += Sig_r[i] * dx[i]

        if i > 0:
            lower[i] = -F[i-1]
            diag[i] += F[i-1]

        if i < N - 1:
            upper[i] = -F[i]
            diag[i] += F[i]

    # Left Boundary Condition (zero incident partial current)
    beta_left = 2.0 * D[0] / dx[0]
    alpha_left = beta_left / (2.0 * beta_left + 1.0)
    diag[0] += alpha_left

    # Right Boundary Condition (zero incident partial current)
    beta_right = 2.0 * D[-1] / dx[-1]
    alpha_right = beta_right / (2.0 * beta_right + 1.0)
    diag[-1] += alpha_right

    A = diags(
        diagonals=[lower[1:], diag, upper[:-1]],
        offsets=[-1, 0, 1],
        format="csr"
    )

    return A


**solve_two_group_k_eigenvalue**

This function takes the matrix built in the function above & solves it iteratively to a certain margin of error (k_tol, source_tol).

An iteration maximum of 500 is applied to allow the code to return an error message, preventing it from entering an infinite loop.

In [79]:
def solve_two_group_k_eigenvalue(regions,k_tol=1e-8,source_tol=1e-8,max_iters=500):
    x, D1, D2, Sa1, Sa2, Ss12, nSf1, nSf2, dx, labels = expand_regions(regions)
    N = len(x)

    # Removal cross sections
    Sr1 = Sa1 + Ss12   # Fast removal: Absorption + Scattering
    Sr2 = Sa2          # Thermal removal: Absorption

    # Building the loss matrices for both groups
    A1 = build_loss_matrix(D1, Sr1, dx)
    A2 = build_loss_matrix(D2, Sr2, dx)

    k = 1.0
    phi1 = np.ones(N)
    phi2 = np.ones(N)

    # Initial fission source
    fission_source = (nSf1 * phi1 + nSf2 * phi2) * dx
    fission_source /= np.sum(fission_source)

    k_history = [k]

    for it in range(1, max_iters + 1):
        # Step 1: Solving fast group
        rhs1 = fission_source / k
        phi1_new = spsolve(A1, rhs1)

        # Step 2: Solving thermal group (source is from scattering from fast group)
        rhs2 = Ss12 * phi1_new * dx
        phi2_new = spsolve(A2, rhs2)

        # Step 3: Calculating new fission source
        fission_source_raw = (nSf1 * phi1_new + nSf2 * phi2_new) * dx
        k_new = k * np.sum(fission_source_raw) / np.sum(fission_source)

        # Step 4: Normalizing fluxes
        normalization_factor = np.sum(fission_source_raw)
        phi1_new /= normalization_factor
        phi2_new /= normalization_factor
        fission_source_new = fission_source_raw / normalization_factor

        # Step 5: Checking for convergence
        k_error = abs(k_new - k) / abs(k)
        source_error = np.linalg.norm(fission_source_new - fission_source, ord=np.inf) / np.linalg.norm(fission_source, ord=np.inf)

        k_history.append(k_new)

        if k_error < k_tol and source_error < source_tol:
            return x, phi1_new, phi2_new, k_new, np.array(k_history), it, nSf1, nSf2, labels
        
        k = k_new
        phi1 = phi1_new
        phi2 = phi2_new
        fission_source = fission_source_new
    
    print("ERROR: Power iteration failed to converge within the maximum number of iterations.")
    return x, phi1_new, phi2_new, k_new, np.array(k_history), it, nSf1, nSf2, labels

**make_region**

In [80]:
def make_region(fuel_dict, length_cm, num_cells, label):
    reg = fuel_dict.copy()
    reg['Nc'] = num_cells
    reg['dx'] = float(length_cm) / float(num_cells)
    reg['label'] = label
    return reg

In [81]:
def plot_reactor_results(x, phi1, phi2, normalized_power, given_core_layout):
    """
    Function built to plot the results of the two-group diffusion solver.
    This is needed so that I can test various different core layouts and compare them in terms of Keff and PPF
    """
    # Plotting results
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 10), sharex=True)

    # Plot 1: Flux Profiles
    ax1.plot(x, phi1, label=r'Fast Flux ($\phi_1$)', color='blue', linewidth=2)
    ax1.plot(x, phi2, label=r'Thermal Flux ($\phi_2$)', color='red', linewidth=2)
    ax1.set_ylabel("Relative Flux")
    ax1.set_title("Axial Fast and Thermal Flux Shapes")
    ax1.grid(True, linestyle='--', alpha=0.6)
    ax1.legend()

    # Color pallete for different fuel types
    default_colors = {
        "fresh": "#d4f1f9", 
        "once": "#f9ebd4", 
        "twice": "#f9d4d4",
        "Reflector": "#e2e2e2"
    }

    # apply background color spans and text labels
    current_x = 0
    for reg in given_core_layout:
        reg_length = reg["dx"] * reg["Nc"]
        label = reg.get("label", "Unknown")
        bg_color = default_colors.get(label, "#f0f0f0") # Fallback to light grey
        
        # background shading
        ax1.axvspan(current_x, current_x + reg_length, color=bg_color, alpha=0.4)
        ax2.axvspan(current_x, current_x + reg_length, color=bg_color, alpha=0.4)
        
        # label text at the top of the bottom plot
        ax2.text(current_x + reg_length/2, 1.02, label, 
                 ha='center', va='bottom', fontsize=9, rotation=0)
        
        current_x += reg_length

    # Plot 2: Normalized Power Distribution
    ax2.plot(x, normalized_power, label="Normalized Power", color='purple', linewidth=2)
    ax2.axhline(1.0, color='black', linestyle='--', label="Average Power", alpha=0.5)
    
    ax2.set_xlabel("Axial Position (cm)")
    ax2.set_ylabel(r"Normalized Power ($P/P_{ave}$)")
    ax2.set_title("Axial Power Distribution")
    ax2.grid(True, linestyle='--', alpha=0.6)
    ax2.legend()

    plt.tight_layout()
    plt.show()
    

In [82]:
def output_results(x, phi1, phi2, k, k_history, iters, nSf1, nSf2, labels, output_core_layout):
    # Calculate Power & Peaking
    power_distribution = nSf1 * phi1 + nSf2 * phi2
    p_ave = np.mean(power_distribution)
    p_max = np.max(power_distribution)
    peaking_factor = p_max / p_ave
    normalized_power = power_distribution / p_ave

    print(f"--- Core Calculation Results ---")
    print(f"k-effective: {k:.5f}")
    print(f"Power Peaking Factor: {peaking_factor:.5f}")
    print(f"Target met? k > 1.05: {k > 1.05}, PPF: < 1.15: {peaking_factor < 1.15}")

    plot_reactor_results(x, phi1, phi2, normalized_power, output_core_layout)


This forms the basis for our two-group diffusion code. Our next step aims to solve the bottom portion of the Part 1 section.

### Radial Derivation

While our initial approach was to use a 1:1:1 ratio alongside one dimension, we found that our power-peaking factor was coming out much higher than we desired. This is because we were doing it wrong.

The problem calls for a 1:1:1 ratio in **three** dimensions, not along the radius in one. and since area increases as a square of radius in a circle, **one must parameterize the radius to produce the correct ratio.**

We begin with three concentric circles, and assume the area to be the following:

$$
A_1 = 1,
A_2 = 2,
A_3 = 3.
$$

**Note that subtracting the previous $A_n$ produces $A_1$**. This will be integral to our radial calculation.

Since these are all equal to $r^2, we can take the square root of each to get the radii:

$$
r_1 = \sqrt{1},
r_2 = \sqrt{2},
r_3 = \sqrt{3}.
$$

Using the relationship above, we can derive the proporional relationship between the rings:

$$
r_{max} = \sqrt{A}.
$$

**Iteratively solving this & subtracting the last will allow us to easily scale the source material's radius as a function of its relative volume.**

This concept will allow us to mathematically translate equal length proportions in one dimension to equal area proportions in two: equal volume, by extension, in our cylindrical geometry.

**radial_core_layout**

In [83]:
'''CONSTANTS'''

fresh =     {"D1": 1.55, "D2": 0.82, "Sa1": 0.0012, "Sa2": 0.0105, "Ss12": 0.0068, "nSf1": 0.00035, "nSf2": 0.0145}
once  =     {"D1": 1.50, "D2": 0.80, "Sa1": 0.0016, "Sa2": 0.0125, "Ss12": 0.0065, "nSf1": 0.00045, "nSf2": 0.0132}
twice =     {"D1": 1.45, "D2": 0.78, "Sa1": 0.0021, "Sa2": 0.0155, "Ss12": 0.0062, "nSf1": 0.00055, "nSf2": 0.0115}

'''PARAMETERS'''

D_regions =   [100,100,100]                     #size of each region, sequentially listed from the center, totaling to 300
b_index   =   ["twice","once","fresh"]          #index for burnup levels

'''FUNCTIONS'''

def radial_core_layout(D_regions,burnups):
    '''
        Takes the equally-sized 1D regions and returns their
        volumetrically proportional counterparts.
    '''
    scaler        = float(300/sqrt(300))        # scaling factor (R=300)
    bookmark      = 0                           # for subtracting r_(n-1)
    loop_no       = 0                           # to force a loop index
    newsize       = []                          # new proportional radius

    for region in D_regions:

        index     = b_index[loop_no]
        loop_no  += 1
        resize    = round((scaler*sqrt(region + bookmark) - scaler*sqrt(bookmark))/2)*2
        bookmark += region
        print("b-index:\t",index,"\nnew radius:\t",resize)
        newsize.append(resize)

    return newsize

print(radial_core_layout(D_regions,b_index))
print(b_index)

final_layout = []

def make_core(newsize,b_index):
    '''
        Takes the volumetrically-proportional regions & their
        burnup levels to produce the final core model.
        returns: region_info
    '''

    region_info     = []
    index           = 0

    for region in newsize:
        print(b_index[index])

        if b_index[index] == "fresh":
            bindex2 = fresh
        if b_index[index] == "once":
            bindex2 = once
        if b_index[index] == "twice":
            bindex2 = twice

        size        = int(newsize[index])
        divs        = int(size/2)
        burnup      = str(b_index[index])
        b_call      = bindex2

        print(b_call)

        make_region(b_call,size,divs,burnup)

        index += 1



    # need a bit to produce reverse region info, then recumulate the sum
    print("b_call:",b_index)
    print("core layout:\t",final_layout)

final_layout = make_core(radial_core_layout(D_regions,b_index),b_index)


# core_layout = [

    # make_region(twice,  55,  27, "twice"),
    # make_region(fresh,   72,  36, "fresh"),
    
    # make_region(once,  346, 173, "once"),
    
    # make_region(fresh,   72,  36, "fresh"),
    # make_region(twice,  55,  27, "twice"),
# ]


# x, phi1, phi2, k, k_history, iters, nSf1, nSf2, labels = solve_two_group_k_eigenvalue(final_layout)
# output_results(x, phi1, phi2, k, k_history, iters, nSf1, nSf2, labels, final_layout)

print(final_layout)

b-index:	 twice 
new radius:	 174
b-index:	 once 
new radius:	 72
b-index:	 fresh 
new radius:	 56
[174, 72, 56]
['twice', 'once', 'fresh']
b-index:	 twice 
new radius:	 174
b-index:	 once 
new radius:	 72
b-index:	 fresh 
new radius:	 56
twice
{'D1': 1.45, 'D2': 0.78, 'Sa1': 0.0021, 'Sa2': 0.0155, 'Ss12': 0.0062, 'nSf1': 0.00055, 'nSf2': 0.0115}
once
{'D1': 1.5, 'D2': 0.8, 'Sa1': 0.0016, 'Sa2': 0.0125, 'Ss12': 0.0065, 'nSf1': 0.00045, 'nSf2': 0.0132}
fresh
{'D1': 1.55, 'D2': 0.82, 'Sa1': 0.0012, 'Sa2': 0.0105, 'Ss12': 0.0068, 'nSf1': 0.00035, 'nSf2': 0.0145}
b_call: ['twice', 'once', 'fresh']
core layout:	 []
None
